# Nettoyage du catalogue Steam

Ce notebook part du fichier brut `data/raw/` et produit un fichier propre dans `data/processed/`. Chaque décision que je vais prendre sera justifiée et chiffrée dans le journal de nettoyage, à la fin de ce notebook.

## 1. Chargement du dataset brut

- Je recharge le fichier d'origine, que je ne modifie jamais à la main.
- Je garde le nombre de lignes de départ dans `nb_depart` pour pouvoir chiffrer tout ce que je retire ensuite.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/steam_games.csv")
nb_depart = len(df)
nb_depart

40833

## 2. Décision 1 : ne garder que les jeux

- Je regarde d'abord la nature des produits de la colonne `types`.

In [2]:
df["types"].value_counts(dropna=False)

types
app       38021
bundle     2572
sub         238
NaN           2
Name: count, dtype: int64

- Je regarde ensuite, pour chaque type, combien de dates de sortie manquent.

In [3]:
df.groupby("types", dropna=False)["release_date"].apply(lambda s: s.isna().sum())

types
app        367
bundle    2572
sub        238
NaN          2
Name: release_date, dtype: int64

- Les `bundle` (2 572) et les `sub` (238) n'ont aucune date de sortie : ce sont des lots de jeux, pas des jeux. Ils fausseraient mes questions sur les prix et sur les sorties par mois. Les 2 valeurs manquantes de `types` ne sont pas des jeux non plus.
- Je ne garde donc que les lignes `app` : 2 812 lignes retirées, il en reste 38 021.

In [4]:
nb_avant = len(df)
df = df[df["types"] == "app"].copy()
print(nb_avant - len(df), "lignes retirées, il reste", len(df))

2812 lignes retirées, il reste 38021


## 3. Décision 2 : les jeux sans nom

- Je compte les jeux dont le nom est manquant : il y en a 14 au total.

In [5]:
df["name"].isna().sum()

np.int64(14)

- Une ligne sans nom ne peut pas être identifiée ni analysée, donc je la supprime : il reste 38 007 jeux.

In [6]:
nb_avant = len(df)
df = df.dropna(subset=["name"]).copy()
print(nb_avant - len(df), "lignes retirées, il reste", len(df))

14 lignes retirées, il reste 38007


## 4. Décision 3 : les doublons

- Je teste combien de lignes ont à la fois le même nom et la même date de sortie : il y en a 4, soit 2 paires.
- Un même nom seul ne suffit pas : 91 lignes partagent un nom avec une autre, mais ce sont surtout des jeux différents (par exemple *Alter Ego* en 2010 et en 1986).

In [7]:
df.duplicated(subset=["name", "release_date"], keep=False).sum()

np.int64(4)

- J'affiche les 4 lignes concernées pour vérifier qu'il s'agit bien de doublons.

In [8]:
brut = pd.read_csv("../data/raw/steam_games.csv")
brut = brut[(brut["types"] == "app") & brut["name"].notna()]
doublons = brut[brut.duplicated(subset=["name", "release_date"], keep=False)]
doublons.sort_values("name")[["name", "release_date", "original_price", "developer"]]

,name,release_date,original_price,developer
1587,Batman: Arkham City - Game of the Year Edition,"Sep 7, 2012",$19.99,"Rocksteady Studios,Feral Interactive (Mac)"
17442,Batman: Arkham City - Game of the Year Edition,"Sep 7, 2012",$19.99,"Rocksteady Studios,Feral Interactive (Mac)"
5410,Warhammer Quest,"Jan 7, 2015",$14.99,"Rodeo Games,Twistplay"
19829,Warhammer Quest,"Jan 7, 2015",NaN,"Twistplay,Rodeo Games"


- *Batman: Arkham City - Game of the Year Edition* et *Warhammer Quest* apparaissent chacun deux fois avec la même date : c'est le même jeu listé deux fois.
- Je supprime ces 2 doublons en gardant la première ligne : il reste 38 005 jeux.

In [9]:
nb_avant = len(df)
df = df.drop_duplicates(subset=["name", "release_date"], keep="first").copy()
print(nb_avant - len(df), "lignes retirées, il reste", len(df))

2 lignes retirées, il reste 38005


- Je vérifie qu'il ne reste plus aucun doublon nom + date : le résultat doit être 0.

In [10]:
df.duplicated(subset=["name", "release_date"], keep=False).sum()

np.int64(0)

## 5. Décision 4 : les prix

- Les prix sont écrits en texte (`$19.99`). Je retire le `$` et je tente de convertir en nombre.
- Ce que je n'arrive pas à convertir devient `NaN` (sans planter), et je compte ces cas.

In [11]:
prix_texte = df["original_price"].str.replace("$", "", regex=False).str.replace(",", "", regex=False)
prix_num = pd.to_numeric(prix_texte, errors="coerce")
non_num = df.loc[prix_num.isna() & df["original_price"].notna(), "original_price"]
len(non_num)

3018

- Je regarde quelles valeurs ne sont pas des prix.

In [12]:
non_num.value_counts().head(10)

original_price
Free              1326
Free To Play       860
Free to Play       529
Demo                32
Free Demo            8
Download Demo        5
DEMO                 4
Free Mod             4
Play for Free!       3
Try Demo             3
Name: count, dtype: int64

- 2 715 valeurs sont des jeux gratuits (`Free`, `Free To Play`, `Free to Play`) : je leur donne le prix 0.
- Les 303 autres sont des démos, des essais ou du texte libre, pas des prix : je les laisse manquantes.
- Je crée une nouvelle colonne `prix` et je garde `original_price` pour pouvoir comparer.

In [13]:
gratuits = ["Free", "Free To Play", "Free to Play"]
df["prix"] = prix_num
df.loc[df["original_price"].isin(gratuits), "prix"] = 0.0

In [14]:
(df["prix"] == 0).sum(), df["prix"].isna().sum()

(np.int64(2715), np.int64(3310))

## 6. Décision 5 : les dates de sortie

- `release_date` est du texte (`May 12, 2016`) : aucun calcul temporel n'est possible dessus.
- Je la convertis en vraie date dans une nouvelle colonne `date_sortie`. Ce qui n'est pas une date complète devient une date manquante (`NaT`).

In [15]:
df["date_sortie"] = pd.to_datetime(df["release_date"], format="%b %d, %Y", errors="coerce")
df["date_sortie"].isna().sum()

np.int64(1943)

- Je sépare les deux causes : les dates absentes au départ et les valeurs qui ne sont pas des dates.

In [16]:
sans_texte = df["release_date"].isna().sum()
pas_une_date = (df["date_sortie"].isna() & df["release_date"].notna()).sum()
sans_texte, pas_une_date

(np.int64(354), np.int64(1589))

- Je regarde ce que contiennent ces valeurs qui ne sont pas des dates.

In [17]:
df.loc[df["date_sortie"].isna() & df["release_date"].notna(), "release_date"].value_counts().head(8)

release_date
2019           296
Coming Soon    134
TBA             74
Summer 2019     68
2020            52
Coming soon     45
TBD             34
Jun 2019        33
Name: count, dtype: int64

- Ces valeurs sont des jeux pas encore sortis ou des dates imprécises. Je garde ces lignes : les analyses sur les prix ou les genres n'ont pas besoin de la date. Seule l'analyse par période les laissera de côté.

In [18]:
df["date_sortie"].min(), df["date_sortie"].max()

(Timestamp('1981-04-22 00:00:00'), Timestamp('2025-04-03 00:00:00'))

## 7. Décision 6 : les valeurs manquantes

- Je regarde d'abord la part de valeurs manquantes de chaque colonne, pour décider colonne par colonne.

In [19]:
(df.isna().mean() * 100).round(1).sort_values(ascending=False)

recent_reviews              92.9
mature_content              92.4
discount_price              69.1
achievements                67.9
minimum_requirements        44.6
recommended_requirements    44.5
desc_snippet                34.7
all_reviews                 25.1
publisher                   13.1
prix                         8.7
original_price               7.9
date_sortie                  5.1
game_details                 1.2
genre                        1.0
release_date                 0.9
developer                    0.8
game_description             0.3
popular_tags                 0.3
name                         0.0
types                        0.0
url                          0.0
languages                    0.0
dtype: float64

### Colonnes supprimées

- `recent_reviews` et `mature_content` sont vides à plus de 92 %, `achievements` à 68 % : trop de manquants pour servir.
- `minimum_requirements`, `recommended_requirements`, `desc_snippet`, `game_description` et `game_details` sont de longs textes que mes cinq questions n'utilisent pas.
- Je supprime ces 8 colonnes.

In [20]:
colonnes_a_supprimer = [
    "recent_reviews", "mature_content", "achievements",
    "minimum_requirements", "recommended_requirements",
    "desc_snippet", "game_description", "game_details",
]
df = df.drop(columns=colonnes_a_supprimer)
df.shape

(38005, 14)

### Lignes sans `genre` ou sans `languages`

- Le genre est la variable centrale de mes questions 1, 2 et 3, et les langues servent à la question 4.
- Seulement 383 jeux n'ont pas de genre et 4 pas de langue, soit environ 1 % : je vais donc supprimer ces lignes.

In [21]:
nb_avant = len(df)
df = df.dropna(subset=["genre", "languages"]).copy()
print(nb_avant - len(df), "lignes retirées, il reste", len(df))

387 lignes retirées, il reste 37618


### Développeur et éditeur inconnus

- Il manque 191 développeurs et 4 713 éditeurs. L'absence est une information (jeux amateurs, petits studios), et je ne peux pas deviner la valeur.
- Je les remplace par "Inconnu" pour ne pas perdre ces lignes.

In [22]:
df["developer"] = df["developer"].fillna("Inconnu")
df["publisher"] = df["publisher"].fillna("Inconnu")
(df["developer"] == "Inconnu").sum(), (df["publisher"] == "Inconnu").sum()

(np.int64(191), np.int64(4713))

## Journal de nettoyage

| Quoi | Pourquoi | Lignes |
|---|---|---|
| Lignes `bundle`, `sub` et `types` manquant supprimées | Lots de jeux ou pages qui ne sont pas des jeux, sans date de sortie | 2 812 |
| Lignes sans `name` supprimées | Jeu non identifiable | 14 |
| Doublons nom + date supprimés | Même jeu listé deux fois | 2 |
| Prix gratuits (`Free`, `Free to Play`) convertis en 0 | Texte sans montant, mais le jeu est gratuit | 2 715 |
| Prix en texte libre (démos, essais) laissés manquants | Ce ne sont pas des prix | 303 |
| `release_date` convertie en vraie date (`date_sortie`) | Le texte ne permet aucun calcul temporel | 36 062 |
| Dates absentes ou incomplètes laissées manquantes | "Coming Soon", "TBA", "Summer 2019"... : pas de date précise | 1 943 |
| 8 colonnes supprimées (`recent_reviews`, `mature_content`, `achievements`, textes longs...) | Plus de 60 % de manquants, ou texte inutile pour mes questions | 38 005 (toutes) |
| Lignes sans `genre` ou `languages` supprimées | Variables centrales des questions 1 à 4, environ 1 % des lignes | 387 |
| `developer` et `publisher` manquants remplacés par "Inconnu" | L'absence est une information, valeur non devinable | 191 + 4 713 |

Après ces 3 étapes/décisions, il reste donc exactement 38 005 lignes.